# Part 8 — Proposed Improvement: QueryBridge and Romanized-Title Retrieval

This notebook connects the literature-derived gap to the implemented improvement and measures it against the frozen baselines. The proposal is a pipeline change, not an unsupported switch to a larger generative model.

**Notebook status:** development-only comparisons; test queries used = 0.


In [1]:
from collections import Counter
from pathlib import Path
import csv
import hashlib
import html
import json
import platform
import random
import statistics

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SEED = 20250816
random.seed(SEED)
FIGURES = ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

def load_json(relative_path):
    return json.loads((ROOT / relative_path).read_text(encoding="utf-8"))

def print_table(rows, columns):
    if not rows:
        print("(no rows)")
        return
    widths = {
        column: max(len(str(column)), *(len(str(row.get(column, ""))) for row in rows))
        for column in columns
    }
    print(" | ".join(str(column).ljust(widths[column]) for column in columns))
    print("-+-".join("-" * widths[column] for column in columns))
    for row in rows:
        print(" | ".join(str(row.get(column, "")).ljust(widths[column]) for column in columns))

def write_bar_svg(filename, values, title, *, maximum=None):
    values = list(values)
    width, left, right, row_height = 820, 245, 80, 34
    height = 76 + row_height * len(values)
    plot_width = width - left - right
    largest = maximum or max((float(value) for _, value in values), default=1.0) or 1.0
    elements = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="white"/>',
        f'<text x="{width / 2}" y="27" text-anchor="middle" font-family="Arial" font-size="18" font-weight="700">{html.escape(title)}</text>',
    ]
    for index, (label, value) in enumerate(values):
        y = 52 + index * row_height
        bar_width = plot_width * float(value) / largest
        elements.extend([
            f'<text x="{left - 10}" y="{y + 17}" text-anchor="end" font-family="Arial" font-size="13">{html.escape(str(label))}</text>',
            f'<rect x="{left}" y="{y}" width="{bar_width:.2f}" height="20" rx="3" fill="#1c5b58"/>',
            f'<text x="{min(left + bar_width + 7, width - 58):.2f}" y="{y + 16}" font-family="Arial" font-size="12">{float(value):.4g}</text>',
        ])
    elements.append('</svg>')
    target = FIGURES / filename
    target.write_text("\n".join(elements) + "\n", encoding="utf-8")
    print(f"Saved visualization: {target.relative_to(ROOT)}")
    return target

print(f"Project: {ROOT.name} | Python: {platform.python_version()} | fixed seed: {SEED}")


Project: h | Python: 3.12.13 | fixed seed: 20250816


## Limitation identified in the baselines

The baseline analysis shows that neither direct semantic similarity nor one deterministic transliteration reliably finds the correct Urdu article. Error categories point particularly to noisy entities, missing vowels, abbreviations, and code switching. Multilingual E5 supplies cross-script semantics, but the reviewed literature warns that dense similarity can confuse closely related entities; transliteration work also shows that orthographic variability cannot always be resolved by one output.

## Improvement design

1. Preserve the original Roman query.
2. Create conservative normalized, Urdu-script, and retrieval-oriented views; reject duplicates and semantic drift.
3. Retrieve every accepted view using title-boosted BM25 and multilingual E5.
4. Romanize each Urdu article title offline and match the original query with character-boundary TF-IDF 2–4 grams. This tolerates missing/inserted vowels and short spelling variation.
5. Contribute only the lead passage for each title match, then combine routes using weighted reciprocal-rank fusion.
6. Optionally rerank a shallow candidate set and apply source/relation-aware evidence gates before answering.

The title route is expected to work because article titles provide concise entity identities while character n-grams avoid requiring an exact transliteration.


In [2]:
bridge = load_json("reports/tables/querybridge_development.json")
baselines = load_json("reports/tables/baselines_development.json")
assert bridge["test_queries_used"] == baselines["test_queries_used"] == 0
comparison = []
for name, values in baselines["systems"].items():
    comparison.append({"system": name, "R@10": values["recall_at_10"], "MRR@10": values["mrr_at_10"], "mean_ms": values["mean_latency_ms"]})
comparison.append({"system": "querybridge_bm25_dense_rrf", "R@10": bridge["result"]["recall_at_10"], "MRR@10": bridge["result"]["mrr_at_10"], "mean_ms": bridge["result"]["mean_latency_ms"]})
print_table(comparison, ["system", "R@10", "MRR@10", "mean_ms"])
print("Mean accepted query views:", bridge["result"]["mean_accepted_variants"])


system                      | R@10     | MRR@10   | mean_ms
----------------------------+----------+----------+--------
direct_dense                | 0.091667 | 0.061528 | 51.788 
single_transliteration_bm25 | 0.025    | 0.00744  | 60.758 
standard_hybrid             | 0.091667 | 0.045602 | 94.522 
querybridge_bm25_dense_rrf  | 0.166667 | 0.074987 | 444.043
Mean accepted query views: 3.467


QueryBridge raises Recall@10 from 0.0917 for the strongest direct/hybrid baselines to 0.1667. This is a useful gain but still leaves five of six relevant passages outside the top ten, so the multi-view bridge alone does not solve entity matching.


## Principal improvement — romanized-title route


In [3]:
regression = load_json("reports/tables/application_accuracy_regression.json")
assert regression["queries"] == 120 and regression["test_queries_used"] == 0
metrics = ["recall_at_1", "recall_at_5", "recall_at_10", "mrr_at_10", "ndcg_at_10"]
rows = [{"metric": metric, "before": f'{regression["before"][metric]:.4f}', "with_title_route": f'{regression["after"][metric]:.4f}', "absolute_gain": f'{regression["after"][metric] - regression["before"][metric]:.4f}'} for metric in metrics]
print_table(rows, ["metric", "before", "with_title_route", "absolute_gain"])
write_bar_svg("proposed_improvement_recall.svg", [("before title route", regression["before"]["recall_at_10"]), ("with title route", regression["after"]["recall_at_10"])], "Proposed improvement: development Recall@10", maximum=1.0)


metric       | before | with_title_route | absolute_gain
-------------+--------+------------------+--------------
recall_at_1  | 0.0750 | 0.3917           | 0.3167       
recall_at_5  | 0.1167 | 0.8750           | 0.7583       
recall_at_10 | 0.1917 | 0.9833           | 0.7917       
mrr_at_10    | 0.1014 | 0.5830           | 0.4817       
ndcg_at_10   | 0.1219 | 0.6796           | 0.5577       
Saved visualization: reports\figures\proposed_improvement_recall.svg


![Proposed improvement](../reports/figures/proposed_improvement_recall.svg)


Recall@10 rises from 0.1917 to 0.9833 on the same 120 title-oriented development questions, an absolute gain of 0.7916. MRR@10 rises from 0.1014 to 0.5830. These figures isolate retrieval before reranking and do **not** mean that 98.33% of unrestricted user answers are correct.


## Worked trace and interface transparency


In [4]:
smoke = load_json("reports/tables/grounded_qa_smoke.json")
trace_rows = [{"type": variant["variant_type"], "accepted": variant["accepted"], "similarity": variant["semantic_similarity"], "reason": variant["decision_reason"]} for variant in smoke["query_variants"]]
print("Query:", smoke["query"])
print_table(trace_rows, ["type", "accepted", "similarity", "reason"])
print("\nSupported:", smoke["supported"])
print("Source:", smoke["source_title"], smoke["source_url"])
print("Evidence sentences:")
for item in smoke["evidence"]:
    print(f'- similarity={item["similarity"]:.4f} | {item["text"]}')
print("Latency (ms):", smoke["latency_ms"])


Query: pakistan ka capital kya hai
type               | accepted | similarity | reason                       
-------------------+----------+------------+------------------------------
original           | True     | 1.0        | accepted_original            
normalized_roman   | False    | 1.0        | rejected_duplicate           
urdu_script        | True     | 0.913508   | accepted_similarity_threshold
retrieval_oriented | True     | 0.900253   | accepted_similarity_threshold

Supported: True
Source: پاکستان کے دارالحکومت https://ur.wikipedia.org/wiki/%D9%BE%D8%A7%DA%A9%D8%B3%D8%AA%D8%A7%D9%86%20%DA%A9%DB%92%20%D8%AF%D8%A7%D8%B1%D8%A7%D9%84%D8%AD%DA%A9%D9%88%D9%85%D8%AA
Evidence sentences:
- similarity=0.8783 | یہ پاکستان کے قومی اور صوبائی دارالحکومتوں کی ایک فہرست ہے۔
- similarity=0.8449 | قومی دار الحکومت 1960ء سے پاکستان کا قومی یا وفاقی دار الحکومت اسلام آباد ہے۔
Latency (ms): {'retrieval': 381.505, 'answer_selection': 1396.865, 'total': 1778.37}


## Leakage controls and remaining risks

Query generation receives only the user's query and supporting transliteration resources—not the gold article, passage ID, evidence, or answer. The title index is built from corpus titles available to every system. Even so, the evaluation questions are title-oriented, so a title route has a structural advantage; independent natural-query evaluation is required. The high-recall retrieval result must be followed by end-to-end answer review and a single locked-test run after configuration freeze.
